<img src="http://wandb.me/logo-im-png" width="400" alt="Weights & Biases" />

<!--- @wandbcode{artifacts-fundamentals} -->


# セットアップ

## 🪄 `wandb` ライブラリのインストールとログイン


まずはライブラリをインストールし、無料アカウントにログインしてください。



## W&B へのログイン
- `wandb login` または `wandb.login()` を使って明示的にログインできます（下記参照）
- 代わりに環境変数を設定することもできます。W&B のロギング動作を変更するために設定できる環境変数がいくつかあります。最も重要なものは次のとおりです:
    - `WANDB_API_KEY` - プロフィール下の「Settings」セクションで確認できます
    - `WANDB_BASE_URL` - W&B サーバーの URL
- API トークンは W&B アプリの「Profile」→「Settings」で確認できます


In [ ]:
import wandb
import os
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(".env", override=True)

YOUR_NAME = os.environ.get("YOUR_NAME")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "SIE-Workshop-2026")
WANDB_HOST = os.environ.get("WANDB_BASE_URL")
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

GENERATED_DIR = Path("notebook_generated_material")
GENERATED_DIR.mkdir(exist_ok=True)
os.environ["WANDB_DIR"] = str(GENERATED_DIR.resolve())

# https://api.wandb.ai はデフォルトで、パブリックホストされたインスタンスを指します
WANDB_HOST = "https://api.wandb.ai" #@param
wandb.login(host= WANDB_HOST)

#Artifacts

W&B Artifacts を使用すると、W&B Run の入力および出力としてデータを追跡・バージョン管理できます。Run にハイパーパラメータ、メタデータ、メトリクスをロギングするのに加えて、Artifact を使用してモデルの学習に使ったデータセットを入力として、結果として得られるモデルチェックポイントを出力としてロギングすることができます。

## データセットの作成
この例で扱うデータセットをいくつか作成しましょう。

In [ ]:
import os
import numpy as np
import csv

directory = str(GENERATED_DIR / "dataset")
os.makedirs(directory, exist_ok=True)
file1, file2 = os.path.join(directory, "file1.csv"), os.path.join(directory, "file2.csv")

def generate_dummy_data(num_samples):
    data = [
        np.random.normal(50, 10, num_samples),
        np.random.randint(1, 100, num_samples),
        np.random.choice(['A', 'B', 'C', 'D'], num_samples),
        np.random.uniform(0.0, 1.0, num_samples)
    ]
    return zip(*data)

def save_to_csv(file, data):
    with open(file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['feature1', 'feature2', 'feature3', 'feature4'])
        writer.writerows(data)

num_samples = 100
save_to_csv(file1, generate_dummy_data(num_samples))
save_to_csv(file2, generate_dummy_data(num_samples))

## Artifact の作成

Artifact を作成する一般的なワークフローは次のとおりです:


1.   Run を初期化する。
2.   Artifact を作成する。
3.   追跡・バージョン管理したい任意のファイルまたはディレクトリを新しい Artifact に追加する。
4.   その Artifact を W&B プラットフォームにロギングする。

これを最も簡単に実現する方法が、以下の例の 2 行目のコードです。これは新しいデータセットを 1 ステップでロギング・追跡・バージョン管理します（つまり、上記の手順 2、3、4 を一度に行います）。

In [ ]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, dir=str(GENERATED_DIR))
run.log_artifact(artifact_or_path=f"{directory}/file1.csv", name=f"my_first_artifact_{YOUR_NAME}", type="dataset")
run.finish()

2 行目では [`run.log_artifact()`](https://docs.wandb.ai/ref/python/public-api/run#log_artifact) を使って Artifact をロギングしています。この例では、関数によく使われる 3 つの引数を渡しています。
1. `artifact_or_path` でバージョン管理したいデータが存在するパスを指定します。ここには任意のファイルまたはディレクトリを追加できます。
2. `name` で、Weights & Biases 内でその Artifact にアクセスするための名前を付けます。
3. `type` で、Artifact により高レベルなグルーピングを付与します。たとえば、type が data の Artifact が複数あったり、type が model の Artifact が複数あったりすることがあります。


その他のよく使われる引数（追加のメタデータを保存する方法など）の詳細については、[Artifacts リファレンス](https://docs.wandb.ai/ref/python/artifact) を参照してください。

上記の `log_artifact` が実行されるたびに、元のデータが変更されていれば wandb は Weights & Biases 内に Artifact の新しいバージョンを作成します。


より細かい制御ができる別のアプローチ（その分、コード行数は増えます）を以下に示します。

In [ ]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, dir=str(GENERATED_DIR))

artifact = wandb.Artifact(f"my_first_artifact_{YOUR_NAME}", type="dataset")
# 以下では 2 つの個別ファイルを Artifact に追加します。
artifact.add_file(local_path=f"{directory}/file1.csv")
artifact.add_file(local_path=f"{directory}/file2.csv")
# ディレクトリ全体の内容を追加したい場合は以下のようにします。
artifact.add_dir(local_path=f"{directory}")
# Artifact を Weights & Biases に明示的にロギングします。
run.log_artifact(artifact)

wandb.finish()

上記の例では、3〜5 行目で Weights & Biases プロジェクト内に新しい Artifact が作成されます。生成された artifact オブジェクトに対して [`artifact.add_file`](https://docs.wandb.ai/ref/python/artifact#add_file) や [`artifact.add_dir`](https://docs.wandb.ai/ref/python/artifact#add_dir) を呼び出すことで、好きなだけファイルやディレクトリを Artifact に追加できます。追加後、Artifact は明示的に Weights & Biases にロギングする必要があります。

## Artifact の使用

特定のバージョンの Artifact をダウンストリームのタスクで使いたい場合は、`v0`、`v1`、`v2` などのように使いたいバージョンを指定するか、自分で追加した特定のエイリアスを指定できます。`latest` エイリアスは常に、ロギングされた Artifact の最新バージョンを指します。

次のコードスニペットでは、W&B Run が `my_first_artifact` という名前で `latest` エイリアスの Artifact を使用するように指定しています:


In [ ]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, dir=str(GENERATED_DIR))
artifact = run.use_artifact(artifact_or_name=f"my_first_artifact_{YOUR_NAME}:latest") # この Run がこの Artifact を使用したという参照が Weights & Biases に作成されます。
path = artifact.download() # Weights & Biases からコードを実行しているローカルシステムに Artifact をダウンロードします。
print(f"Data directory located at {path}")
# ダウンロードした Artifact を使って学習を実行
run.finish()

コマンドラインからのダウンロード方法を含む、Artifact のダウンロードをカスタマイズする方法の詳細については、[ダウンロードと利用ガイド](https://docs.wandb.ai/guides/artifacts/download-and-use-an-artifact) を参照してください。

## Artifact の新しいバージョンの作成

ここでは、変更を追跡・バージョン管理しながら、データセットを修正したいとします。次の例では、データセットをサブサンプリングして新しいファイルとして保存します。CSV ファイルの読み込みには [Pandas](https://pandas.pydata.org/pandas-docs/stable/index.html) ライブラリを使用します。

2 つ目のコードブロックでは、同じ Artifact 名（*my_first_artifact*）で Weights & Biases にロギングし、これが既存の Artifact の新しいバージョンであることを Weights & Biases に認識させます。

In [ ]:
import pandas
df = pandas.read_csv(f"{directory}/file1.csv")
# 元のサイズの 50% にサブサンプリング
df_subsampled = df.sample(frac=0.5, random_state=1)
# サブサンプリングした DataFrame を新しいファイルとして保存
df_subsampled.to_csv(f"{directory}/file1.csv", index=False)

ローカルにサブサンプリングされた新しいバージョンのデータセットが用意できたので、新しいバージョンを Weights & Biases にロギングできます。

In [ ]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, dir=str(GENERATED_DIR))
run.log_artifact(artifact_or_path=f"{directory}/file1.csv", name=f"my_first_artifact_{YOUR_NAME}", type="dataset", aliases =["subsampled"])
run.finish()

これでサブサンプリングされたデータセットが `my_first_artifact_<name>` Artifact の新しいバージョンとしてロギングされます。

この Artifact にはカスタムの `alias` も付与されています。`alias` は、この Artifact バージョンに対する一意のラベルです。現在の `alias` は `subsampled` ですが、デフォルトのエイリアスは `vN` で、`N` は Artifact のバージョン数を表します。これは自動的にインクリメントされます。エイリアスを使えば、いつでも Artifact の特定のバージョンにアクセスできます。

## Artifact バージョンのメタデータの更新

W&B プラットフォーム上で、W&B Run の内側でも外側でも、Artifact の `description`、`metadata`、`alias` を更新することができます。


次の例では、Run の内側で `my_first_artifact` Artifact の `description` を変更します:

In [ ]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, dir=str(GENERATED_DIR))
artifact = run.use_artifact(artifact_or_name=f"my_first_artifact_{YOUR_NAME}:subsampled")
artifact.description = "This is an edited description."
artifact.metadata = {"source": "local disk", "internal data owner": "platform team"}
artifact.save()  # Artifact のプロパティへの変更を永続化します
run.finish()

## パイプライン内での Artifact の利用
Artifact が Weights & Biases で追跡・バージョン管理されると、ML ワークフローに統合するのが簡単になります。

In [ ]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, dir=str(GENERATED_DIR))
artifact = run.use_artifact(artifact_or_name=f"my_first_artifact_{YOUR_NAME}:latest")
# 以下は読者への演習として残しておきます
# モデルの学習
# モデルを Artifact としてロギング
run.finish()


## Artifacts UI のナビゲーション

W&B プラットフォームを通じて Artifact を管理することもできます。これにより、モデルのパフォーマンスやデータセットのバージョン管理に関する洞察を得ることができます。関連する情報に移動するには、こちらの [リンク](https://wandb.ai/wandb-smle/artifact_workflow/overview) をクリックし、**Artifacts** タブをクリックしてください。

タブ内の **Lineage** セクションに移動すると、`run.use_artifact()` を呼び出して Artifact を Run の入力にした場合、`run.log_artifact()` を呼び出して Artifact を Run の出力にした場合に形成される依存グラフが表示されます。これにより、プロジェクト内のさまざまなモデルバージョンとデータセットやジョブといった他のオブジェクトとの関係を可視化できます。プロジェクトのリネージページに移動するには [こちら](https://wandb.ai/wandb-smle/artifact_workflow/artifacts/dataset/preprocessed/v6/lineage) のリンクをクリックしてください。

## **Artifacts の Time-to-live (TTL)**

W&B Artifacts は、Artifact の各バージョンに time-to-live ポリシーを設定することをサポートしています。以下の例では、一般的な Artifact ロギングのワークフローにおける TTL ポリシーの使用方法を示します。次の内容をカバーします:

* Artifact を作成するときに TTL ポリシーを設定する
* 特定の Artifact エイリアスに対して遡って TTL を設定する


## 新しい Artifact に TTL を設定する
以下では、2 つの小さな MNIST CSV ファイルをダウンロードし、`mnist_dataset` 型の Artifact としてアップロードした上で TTL を割り当てます。
- mnist_test.csv
- mnist_train_small.csv


In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

sample_data_dir = GENERATED_DIR / "sample_data"
sample_data_dir.mkdir(exist_ok=True)

mnist = fetch_openml("mnist_784", version=1, as_frame=True, parser="auto")
df = mnist.frame

# 小さな学習セット（先頭 10,000 行）
df.head(10000).to_csv(sample_data_dir / "mnist_train_small.csv", index=False)

# テストセット（末尾 10,000 行）
df.tail(10000).to_csv(sample_data_dir / "mnist_test.csv", index=False)

print(f"MNIST CSVs saved to {sample_data_dir}/")

In [ ]:
from datetime import timedelta

run = wandb.init(entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                job_type="raw-data",
                dir=str(GENERATED_DIR))

raw_mnist_train = wandb.Artifact(
    f"mnist_train_small_{YOUR_NAME}",
    type="mnist_dataset",
    description="Small MNIST Training Set"
)

# add_reference はファイルをアップロードせずに、パスとチェックサムでファイルを追跡します
# （データはローカルのまま、リネージだけを追跡するためワークショップのネットワーク負荷を軽減できます）
train_csv = GENERATED_DIR / "sample_data" / "mnist_train_small.csv"
raw_mnist_train.add_reference(f"file://{train_csv.resolve()}")
raw_mnist_train.ttl = timedelta(days=10)
run.log_artifact(raw_mnist_train, aliases=["small", "mnist", "train"])

raw_mnist_test = wandb.Artifact(
    f"mnist_test_small_{YOUR_NAME}",
    type="mnist_dataset",
    description="Small MNIST Test Set"
)

test_csv = GENERATED_DIR / "sample_data" / "mnist_test.csv"
raw_mnist_test.add_reference(f"file://{test_csv.resolve()}")
raw_mnist_test.ttl = timedelta(days=10)
run.log_artifact(raw_mnist_test, aliases=["small", "mnist", "test"])

run.finish()


特定の Artifact エイリアスに対して遡って TTL を設定する

In [ ]:
from datetime import timedelta

run = wandb.init(entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                job_type="modify-ttl",
                dir=str(GENERATED_DIR))

test_art = run.use_artifact(f"{WANDB_ENTITY}/{WANDB_PROJECT}/mnist_test_small_{YOUR_NAME}:latest")
test_art.ttl = timedelta(days=365)  # 1 年後に削除
test_art.save()

train_art = run.use_artifact(f"{WANDB_ENTITY}/{WANDB_PROJECT}/mnist_train_small_{YOUR_NAME}:latest")
train_art.ttl = timedelta(days=2)  # 2 日後に削除
train_art.save()

print(test_art.ttl)
print(train_art.ttl)

run.finish()

## Artifact References

Artifact は現在、次の URI スキームをサポートしています:

* **http(s)://:** HTTP 経由でアクセス可能なファイルへのパス。HTTP サーバーが ETag および Content-Length レスポンスヘッダーをサポートしている場合、Artifact はチェックサム（etag 形式）とサイズメタデータを追跡します。
* **s3://:** S3 のオブジェクトまたはオブジェクトプレフィックスへのパス。Artifact は参照されるオブジェクトのチェックサムとバージョニング情報（バケットでオブジェクトバージョニングが有効な場合）を追跡します。オブジェクトプレフィックスは、そのプレフィックス配下のオブジェクトを含むように展開されます（デフォルトで最大 100,000 オブジェクト）。
* **gs://:** GCS のオブジェクトまたはオブジェクトプレフィックスへのパス。Artifact は参照されるオブジェクトのチェックサムとバージョニング情報（バケットでオブジェクトバージョニングが有効な場合）を追跡します。オブジェクトプレフィックスは、そのプレフィックス配下のオブジェクトを含むように展開されます（デフォルトで最大 100,000 オブジェクト）。

ローカルファイルを参照する例を以下に示します。

In [ ]:
run = wandb.init(entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                job_type="upload-references",
                dir=str(GENERATED_DIR))
artifact = wandb.Artifact(name=f"local-file-references_{YOUR_NAME}", type="reference-dataset")
artifact.add_reference(f"file://{GENERATED_DIR.resolve()}", checksum=True)
run.log_artifact(artifact)
run.finish()

## **Artifacts のキャッシュおよびステージングディレクトリに関する考慮事項**

デフォルトでは、wandb Artifacts は次のワークフローのために、サイズの 2 倍のストレージ容量が必要になります。

Artifact をアップロードする際、wandb はローカルに 2 つのコピーを作成します。ひとつは `.cache directory`（`use_artifacts` 呼び出し時にファイルをダウンロードせずに高速に取得するために使われます）、もうひとつは `.local/share/wandb/artifacts/staging` で、アップロード中にファイルが更新された場合に問題が生じないよう、ファイルが複製されます。ステージングが存在する理由は、追加したファイルがアップロード前に変更されることを防ぐためです。たとえば、`artifact.add(file.txt)` を呼び出した後にコード内で `file.txt` を変更した場合でも、W&B は追加した時点での `file.txt` のオリジナル内容をアップロードすることを現在保証しています。

Artifact のキャッシュ先を指定するには？
- ユーザーが読み書き権限を持つディレクトリに対して `WANDB_CACHE_DIR` 環境変数を設定します。

Artifact のステージング先を指定するには？
- ユーザーが読み書き権限を持つディレクトリに対して `WANDB_DATA_DIR` 環境変数を設定します。

キャッシュおよびステージングは、`add_dir` および `add_file` メソッドで `skip_cache` と `policy` を設定することにより、ユーザーが直接制御することもできます。

考えられるすべての組み合わせを以下に示します:

- `skip_cache=True` かつ `policy=mutable`: ステージングのみが作成されます。
- `skip_cache=False` かつ `policy=mutable`: ステージングとキャッシュファイルが作成されますが、キャッシュ中にステージングファイルは削除されます。
- `skip_cache=True` かつ `policy=immutable`: ステージングもキャッシュファイルも作成されません。
- `skip_cache=False` かつ `policy=immutable`: キャッシュファイルのみが作成されます。

# Artifact を使った学習パイプラインの例

以下のパイプラインには次の内容が含まれます:

* データのバージョン管理: Heart Disease データセットを学習・検証・テストセットに分割し、それぞれを W&B の Artifact としてロギングして、追跡と再現性を容易にします。

* モデル学習: ニューラルネットワークを学習セットで学習させ、検証セットでパフォーマンスをモニタリングします。最良のモデルバージョンを W&B の Artifact として保存・バージョン管理します。

* モデル評価: W&B から最良のモデルを取得し、テストセットで評価することで、W&B が ML ライフサイクル全体で再現性とトレーサビリティを確保する仕組みを示します。

* 紹介される内容:
W&B を使ったシームレスなデータおよびモデルのバージョン管理の方法。
学習および評価中にモデルパフォーマンスを追跡するためのベストプラクティス。
プロダクション対応環境での効率的かつ再現可能な ML ワークフロー。

### データの準備と wandb へのアップロード

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Heart Disease データセットを読み込み
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
data = pd.read_csv(url, header=None, names=columns)

# 欠損値 ('?') を NaN に置き換え、NaN を含む行を削除
data.replace('?', np.nan, inplace=True)
data = data.dropna().astype(float)

# ターゲット変数を変換: 0 = 心疾患なし、1 = 心疾患あり
data['target'] = (data['target'] > 0).astype(int)

# データセットをシャッフルしてランダムな分布を確保
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# 学習/検証/テストに分割 (60/20/20)
train_size = int(0.6 * len(data))
val_size = int(0.2 * len(data))
test_size = len(data) - train_size - val_size

train_data = data[:train_size]
val_data = data[train_size:train_size + val_size]
test_data = data[train_size + val_size:]

# データセット全体を CSV ファイルとして保存
data.to_csv(str(GENERATED_DIR / "heart_disease_full_dataset.csv"), index=False)

# データセット Artifact を保存・ロギングするためのシンプルな関数
def save_and_log_dataset(data, filename, artifact_name, aliases):
    # データセットを CSV ファイルとして保存
    data.to_csv(filename, index=False)

    # データセット Artifact を作成してロギング
    dataset_artifact = wandb.Artifact(name=artifact_name, type='dataset')
    # Reference artifact: アップロードせずに、パスとチェックサムでファイルを追跡
    # （データはローカルに残り、W&B はメタデータのみを記録するためネットワーク負荷が小さい）
    dataset_artifact.add_reference(f"file://{os.path.abspath(filename)}")
    wandb.log_artifact(dataset_artifact, aliases=aliases)

# データを reference artifact としてロギング（アップロードなし、パス＋チェックサムを追跡）
run = wandb.init(entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                group = YOUR_NAME,
                job_type="heart-disease-data-uploads",
                name = f"heart_disease_data_uploads_{YOUR_NAME}",
                tags = ["data-upload"],
                dir=str(GENERATED_DIR),
                )

# データセット全体を保存・ロギング
save_and_log_dataset(data, str(GENERATED_DIR / "heart_disease_full_dataset.csv"), f'heart_disease_full_dataset_{YOUR_NAME}', ["initial_commit", "complete_dataset"])

# 学習データセットを保存・ロギング
save_and_log_dataset(train_data, str(GENERATED_DIR / "heart_disease_train_dataset.csv"), f'heart_disease_train_dataset_{YOUR_NAME}', ["initial_commit", "train_split"])

# 検証データセットを保存・ロギング
save_and_log_dataset(val_data, str(GENERATED_DIR / "heart_disease_val_dataset.csv"), f'heart_disease_validation_dataset_{YOUR_NAME}', ["initial_commit", "validation_split"])

# テストデータセットを保存・ロギング
save_and_log_dataset(test_data, str(GENERATED_DIR / "heart_disease_test_dataset.csv"), f'heart_disease_test_dataset_{YOUR_NAME}', ["initial_commit", "test_split"])


# 全データセットを W&B テーブルにロギングして視覚的に分析できるようにする
wandb.log({f"train_data_table_{YOUR_NAME}": wandb.Table(dataframe=train_data),
           f"test_data_table_{YOUR_NAME}": wandb.Table(dataframe=test_data),
           f"validation_data_table_{YOUR_NAME}": wandb.Table(dataframe=val_data)})

wandb.finish()

### モデル学習と Artifact のロギング
以下の例で行う内容は次のとおりです

1. `load_data` の中で artifact.download() を使って、学習用・検証用データセット全体をダウンロードします
2. 学習データを使って学習 Run を実行し、`val_loss < best_performance` の条件を満たす場合に、新しいモデル Artifact を作成して `best` エイリアス付きでロギングします
3. その後、`best` モデルをテストデータセットに対して検証します

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# wandb から Artifact を読み込むためのシンプルな関数
def load_data(entity, project, artifact_name, your_name, split_name):

    artifact_full_name = f'{entity}/{project}/{artifact_name}_{your_name}:latest'
    artifact = wandb.use_artifact(artifact_full_name, type='dataset')
    artifact_dir = artifact.download()# Artifact のローカル版を使う場合は、ダウンロードせずに wandb.use_artifact() のみを利用して Artifact にリネージを紐付けることもできます

    data = pd.read_csv(f"{artifact_dir}/heart_disease_{split_name}_dataset.csv")

    X = torch.tensor(data.drop("target", axis=1).values, dtype=torch.float32)
    y = torch.tensor(data["target"].values, dtype=torch.float32)

    return X, y

# 学習用 Run を初期化
run = wandb.init(entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                group = YOUR_NAME,
                job_type="heart-disease-training",
                name = f"heart_disease_training_validation_{YOUR_NAME}",
                dir=str(GENERATED_DIR),
                )

# 学習データを読み込み
X_train, y_train = load_data(WANDB_ENTITY, WANDB_PROJECT, 'heart_disease_train_dataset', YOUR_NAME, 'train')

# 検証データを読み込み
X_val, y_val = load_data(WANDB_ENTITY, WANDB_PROJECT, 'heart_disease_validation_dataset', YOUR_NAME, 'val')

# シンプルなニューラルネットワークモデルを定義
class HeartDiseaseModel(nn.Module):
    def __init__(self, input_size):
        super(HeartDiseaseModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.fc3 = nn.Linear(64, 32)
        self.bn3 = nn.BatchNorm1d(32)
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)
        x = F.relu(self.fc4(x))
        x = torch.sigmoid(self.fc5(x))
        return x

model = HeartDiseaseModel(input_size=X_train.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

best_performance = float('inf')
version = 1

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    # 学習精度を計算してロギング
    predictions = (outputs >= 0.5).float()
    train_accuracy = (predictions == y_train).float().mean().item()

    wandb.log({"train/epoch": epoch, "train/train_loss": loss.item(), "train/train_accuracy": train_accuracy})

    # 検証セットでモデルを評価
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val).squeeze()
        val_loss = criterion(val_outputs, y_val).item()

        # 検証精度を計算してロギング
        val_predictions = (val_outputs >= 0.5).float()
        val_accuracy = (val_predictions == y_val).float().mean().item()

        wandb.log({"val/val_loss": val_loss, "val/val_accuracy": val_accuracy})

        if val_loss < best_performance:
            best_performance = val_loss
            model_path = str(GENERATED_DIR / f"heart_disease_model_v{version}.pth")
            torch.save(model.state_dict(), model_path)
            artifact = wandb.Artifact(name=f'heart_disease_model_{YOUR_NAME}', type='model')
            # Reference artifact: アップロードせずに、パスとチェックサムでモデルを追跡
            artifact.add_reference(f"file://{os.path.abspath(model_path)}")
            wandb.log_artifact(artifact, aliases=[f"v{version}", "best"])
            version += 1

wandb.finish()


### バージョン管理されたモデル Artifact を使った評価フェーズ
学習ループでロギングした最良のモデルを使って、テストデータセットに対して評価を行います。

In [ ]:
# 評価用に新しい W&B Run を開始
run = wandb.init(entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                group=YOUR_NAME,
                job_type="heart-disease-testing",
                name=f"heart_disease_testing_{YOUR_NAME}",
                dir=str(GENERATED_DIR),
                )

# テストデータを読み込み
X_test, y_test = load_data(WANDB_ENTITY, WANDB_PROJECT, 'heart_disease_test_dataset', YOUR_NAME, 'test')

# テストセットで評価を実行
with torch.no_grad():
    outputs = model(X_test).squeeze()
    test_loss = criterion(outputs, y_test).item()
    wandb.log({"test/final_test_loss": test_loss})

wandb.finish()



# Registry

W&B Registry は、アセットに対するバージョニング、エイリアス、リネージ追跡、ガバナンスを提供する、キュレーションされた中央リポジトリです。Registry を使用すると、組織全体の個人やチームが、すべてのモデル、データセット、その他の Artifact のライフサイクルを共有し、共同で管理することができます。Registry には SaaS では https://wandb.ai/registry から直接アクセスでき、プライベートインスタンスでは `https://<host-url>/registry` からアクセスできます。

Registry には `Model` と `Datasets` という 2 つのコアレジストリがあり、ユーザーは独自のカスタムレジストリを作成することもできます。

Registry のユースケース例:

- モデル以外のオブジェクト（予測値やデータセットなど）が、別プロジェクトで公開されたモデルを使うプロダクション Run でも有効であることを示すためのレジストリ。
- モデルや実験に留まらず、データセットやプロンプトなどの要素も含めた包括的な追跡。
- 「データセット製造」チームから、中央で発見可能なリストへとデータセットを引き渡すことを可能にします。Registry に昇格されると、適切な承認を得た任意のチームがこれらのデータセットをモデル学習に使用できます。
- モデルが開発されたプロジェクト／チームへのアクセスを付与することなく、組織全体でモデルを共有することを可能にします。
- モデルレジストリは、チームがトレーサビリティを持ってモデルをロギング・利用することをサポートしますが、より大規模な組織では、異なるロール間でのコラボレーションや共有強化が必要になります。




**ハンズオン例:** メインのワークショップノートブック (`aqua_with_wandb.ipynb`) では、ベースラインのステージング、スイープ結果との比較、CI/CD 自動化によるプロダクションへの勝者の昇格といった、Registry ワークフロー全体を順を追って説明します。

In [ ]:
# import shutil

# if GENERATED_DIR.exists():
#     shutil.rmtree(GENERATED_DIR)
#     print(f"クリーンアップ完了: {GENERATED_DIR}/")
# else:
#     print(f"クリーンアップ対象なし — {GENERATED_DIR}/ は存在しません。")

# # ノートブックフォルダ内に残った wandb/ ディレクトリも削除
# from pathlib import Path
# wandb_dir = Path("wandb")
# if wandb_dir.exists():
#     import shutil
#     shutil.rmtree(wandb_dir)
#     print(f"クリーンアップ完了: {wandb_dir}/")
# else:
#     print(f"{wandb_dir}/ ディレクトリは見つかりませんでした。")

# # ノートブックフォルダ内に残った artifact/ ディレクトリも削除
# artifacts_dir = Path("artifacts")
# if artifacts_dir.exists():
#     import shutil
#     shutil.rmtree(artifacts_dir)
#     print(f"クリーンアップ完了: {artifacts_dir}/")
# else:
#     print(f"{artifacts_dir}/ ディレクトリは見つかりませんでした。")


# リソース

**Artifacts**
- [Lineage](https://docs.wandb.ai/guides/artifacts/explore-and-traverse-an-artifact-graph): W&B Artifact システム使用時に自動的に構築されるリネージグラフを表示します。特定の Artifact バージョン、データセット、モデル、Run の関係を監査可能な形で視覚的に概観できます。
- [Artifact Automations](https://docs.wandb.ai/models/automations/automation-events#project): 学習データの新しいバージョンがロギングされるたびに新しいモデルを自動的に学習するなど、Artifact の変更に基づいて特定の Weights & Biases ジョブを自動的に実行します。
- [Reference Artifacts](https://docs.wandb.ai/guides/artifacts/track-external-files#download-a-reference-artifact): Amazon S3 バケット、GCS バケット、Azure Blob など、W&B サーバー外に保存されたファイルを追跡します。
- [Artifact TTL](https://docs.wandb.ai/guides/artifacts/ttl): W&B Artifact time-to-live (TTL) ポリシーを使って、Artifact が W&B から削除されるタイミングをスケジュールします。

**Registry**

- [Registry overview](https://docs.wandb.ai/guides/registry/)
- [Link a version to a collection](https://docs.wandb.ai/guides/registry/link_version)
- [Automations & webhooks](https://docs.wandb.ai/models/automations)